# 📝 과제 01: 낙서 맞히기 CNN

## 만들 결과

Quick, Draw!는 사람이 20초 안에 그린 낙서를 모은 데이터셋입니다. 그 가운데 10종(고양이, 자전거, 피자, 사과, 집, 별, 물고기, 우산, 시계, 비행기)을 교안 01의 CNN으로 분류하고, 어떤 낙서끼리 헷갈리는지 찾습니다.

<img src="images/80_quickdraw_overview.png" width="1050" alt="제목은 '낙서 한 장에서 10종의 로짓과 예측까지'입니다. 왼쪽에 고양이 낙서가 든 흑백 격자 '(1, 28, 28)'이 있고, 오른쪽으로 파란 상자 '합성곱 + 풀링', 얇은 판 여러 장, 다시 '합성곱 + 풀링', 더 작은 판, 회색 상자 'Flatten', 초록 상자 'Linear'가 이어집니다. 맨 오른쪽 '로짓 10개' 가로 막대는 위에서부터 '고양이', '자전거', '피자', '사과', '집', '별', '물고기', '우산', '시계', '비행기'이고, '고양이' 막대만 주황으로 가장 길며 말풍선 '예측: 고양이'가 붙어 있습니다. 아래 띠에 '로짓이 가장 큰 종류가 예측입니다'가 적혀 있습니다.">

- 종류 순서로 쌓인 낙서를 섞고 CNN 입력 모양 `(N, 1, 28, 28)`로 바꿉니다.
- 필터 수를 받아 CNN을 만들고, 학습 전 첫 손실이 log 10(2.30) 근처인지 확인합니다.
- 필터 수 8, 16, 32개 후보를 학습 데이터 앞 3,000장으로 학습해 학습 데이터 정확도를 비교합니다.
- 필터 32개로 최종 모델을 학습하고, 테스트 정확도(약 0.91)를 한 번 계산해 학습 데이터 정확도와 비교합니다.
- 가장 많이 헷갈린 종류 짝(별을 비행기로 예측, 17장)을 찾고, 모델을 저장해 다시 불러옵니다.
- 둘째 합성곱 층과 입출력 크기가 같은 `nn.Linear`의 파라미터 수를 비교합니다(4,640개와 19,675,264개).
- 합성곱마다 배치 정규화를 넣은 CNN을 같은 조건으로 학습해 원래 CNN과 학습 손실을 비교합니다(5에폭 뒤 0.219와 0.442).

## 준비: 데이터와 제공 코드

- **자료:** `data/과제01/quickdraw10.pt`. Google Quick, Draw! 데이터셋(CC BY 4.0)에서 10종을 골랐고, 종류마다 학습 1,000장, 테스트 200장입니다. 그림은 28x28 흑백(0~255 정수)이고 **종류 순서로 쌓여 있습니다**(0번부터 999번까지 고양이, 1,000번부터 자전거, ...).
- **입력과 정답:** 입력은 낙서 그림 한 장, 정답은 종류 번호(0~9)입니다.
- **제공 코드:** 준비 셀이 아래 이름과 지난 시간 교안의 `train_one_epoch`, `evaluate`를 만듭니다.

| 이름 | 내용 |
|---|---|
| `quickdraw` | 위 파일을 읽은 dict. 키는 `train_images`(10,000장), `train_labels`, `test_images`(2,000장), `test_labels`, `class_names` |
| `class_names` | 종류 이름 10개 |
| `output_dir` | 저장한 모델 파일을 둘 `output` 폴더 |

- **작성:** `[작성]` 셀의 `...`를 채우고 바로 아래 `[자가채점]` 셀을 실행하세요. 뒤 문항이 앞 문항의 변수를 쓰므로 1번부터 차례로 풉니다.
- **무작위로 고를 때의 정확도:** 종류가 10개라서 아무것도 배우지 못한 모델의 정확도는 0.1 안팎, 손실은 log 10 = 2.30 근처입니다.
- **실행 시간:** CPU에서 과제 전체가 수 분 안에 끝납니다.

In [ ]:
import platform
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
import torch

material_dir = Path(".")  # data가 있는 단원 폴더. 정답 노트북에서는 ".."
data_dir = material_dir / "data"

# 계산 장치: RunPod처럼 GPU(cuda)가 있으면 GPU, 없으면 CPU에서 계산합니다
if torch.cuda.is_available():
    device = torch.device("cuda")
    print("계산 장치:", device, "|", torch.cuda.get_device_name(0))
else:
    device = torch.device("cpu")
    print("계산 장치:", device)

# 그래프 글꼴: 영문, 숫자, 기호는 DejaVu Sans, 한글은 운영체제의 한글 글꼴이 맡습니다
from matplotlib import font_manager

if platform.system() == "Windows":
    korean_font = "Malgun Gothic"
elif platform.system() == "Darwin":
    korean_font = "AppleGothic"
else:
    korean_font = "NanumGothic"
    # 리눅스(RunPod): 글꼴을 설치하기 전에 만든 matplotlib 글꼴 목록에는 나눔고딕이 없으므로, 설치된 파일을 직접 등록합니다
    for font_path in font_manager.findSystemFonts():
        if "NanumGothic" in font_path:
            font_manager.fontManager.addfont(font_path)
plt.rcParams["font.family"] = ["DejaVu Sans", korean_font]
plt.rcParams["axes.unicode_minus"] = False  # 음수 부호가 네모로 깨지지 않게 합니다

In [ ]:
# 낙서 10종: 학습 10,000장, 테스트 2,000장
quickdraw = torch.load(data_dir / "과제01" / "quickdraw10.pt")
class_names = quickdraw["class_names"]
print("종류:", class_names)
print("학습 그림:", tuple(quickdraw["train_images"].shape), "| 테스트 그림:", tuple(quickdraw["test_images"].shape))

# 저장한 모델 파일을 둘 폴더
output_dir = material_dir / "output"
output_dir.mkdir(exist_ok=True)

In [ ]:
# 종류마다 첫 낙서 한 장씩
fig, axes = plt.subplots(2, 5, figsize=(14, 6))
for c in range(10):
    # 종류 순서로 쌓여 있으므로 c번 종류의 첫 그림은 c * 1000번입니다
    ax = axes[c // 5, c % 5]
    ax.imshow(quickdraw["train_images"][c * 1000], cmap="gray_r")
    ax.set_title(class_names[c], fontsize=14)
    ax.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
# 지난 시간 교안에서 만든 학습 함수와 평가 함수를 그대로 씁니다
def train_one_epoch(model, loader, loss_fn, optimizer, scheduler=None):
    """미니배치마다 파라미터를 한 번씩 갱신하고, 그 에폭 미니배치 손실의 평균을 돌려줍니다."""
    total_loss = 0.0
    for X_batch, y_batch in loader:
        # 미니배치를 모델이 있는 계산 장치로 옮깁니다
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        # 지난 스텝의 기울기를 지우고, 순전파 -> 손실 -> 역전파 -> 갱신
        optimizer.zero_grad()
        loss = loss_fn(model(X_batch), y_batch)
        loss.backward()
        optimizer.step()

        # 스케줄러가 있으면 스텝마다 학습률을 바꿉니다
        if scheduler is not None:
            scheduler.step()

        total_loss += loss.item()

    # len(loader)는 미니배치 수입니다
    return total_loss / len(loader)


def evaluate(model, loader, loss_fn):
    """기울기 없이 예측해 (평균 손실, 정확도)를 돌려줍니다."""
    total_loss = 0.0
    correct = 0
    with torch.no_grad():
        for X_batch, y_batch in loader:
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)
            logits = model(X_batch)

            # 마지막 미니배치는 장 수가 적으므로, 손실에 장 수를 곱해 더한 뒤 전체 장 수로 나눕니다
            total_loss += loss_fn(logits, y_batch).item() * len(y_batch)
            correct += (logits.argmax(dim=1) == y_batch).sum().item()

    n_samples = len(loader.dataset)
    return total_loss / n_samples, correct / n_samples

## 1. 낙서를 섞고 CNN 입력 모양으로 바꾸기

**배경**: 학습 데이터가 종류 순서로 쌓여 있어서, 섞지 않고 앞에서 잘라 쓰면 고양이, 자전거, 피자 세 종류만 나옵니다. 5번에서 앞 3,000장으로 후보를 비교하므로 먼저 한 번 섞고, CNN이 받는 `(장 수, 채널, 높이, 너비)` 모양으로 바꿉니다.

<img src="images/81_stacked_vs_shuffled.png" width="1050" alt="제목은 '앞에서 잘라 쓰기 전에 먼저 섞습니다'입니다. 위아래 두 줄이 있습니다. 윗줄 '섞기 전'에는 긴 띠가 10칸 색 구역(구역마다 다른 색)으로 나뉘어 왼쪽부터 '고양이', '자전거', '피자', '사과', '집', '별', '물고기', '우산', '시계', '비행기' 순서로 이어지고, 띠 왼쪽 3칸을 덮는 괄호에 'X_train[:3000]: 고양이, 자전거, 피자만'이 적혀 있습니다. 아랫줄 '섞은 뒤'에는 같은 열 가지 색의 가는 줄무늬가 띠 전체에 고르게 섞여 있고, 왼쪽 3칸 너비를 덮는 괄호에 'X_train[:3000]: 10종이 약 300장씩'이 적혀 있습니다. 두 줄 사이 아래 방향 화살표에 '그림과 정답을 같은 순서로 섞기'가 적혀 있습니다.">

**요구사항**
- (1) 시드 0 generator(제공 줄)로 학습 그림 10,000장을 섞는 순서 **`order`**(정수 텐서, 10,000개)를 만듭니다.
- (2) 학습 그림을 `order` 순서로 꺼내 0~1 실수(float32)로 바꾸고 채널 축을 더한 **`X_train`**(10000, 1, 28, 28)을 만듭니다.
- (3) 학습 정답을 같은 `order` 순서로 꺼낸 **`y_train`**(정수 텐서, 10,000개)을 만듭니다.
- (4) 테스트 그림을 섞지 않고 같은 방법으로 모양만 바꾼 **`X_test`**(2000, 1, 28, 28)를 만듭니다.
- (5) 테스트 정답 **`y_test`**(정수 텐서, 2,000개)를 섞지 않고 만듭니다.

**예시**: 섞기 전 앞 3,000장은 고양이 1,000, 자전거 1,000, 피자 1,000이고 나머지 7종은 0장입니다. 섞은 뒤에는 10종 모두 287~311장입니다.

**확인 기준**: `order`는 0~9999가 한 번씩 들어 있고 앞 20개가 시드 0 순서와 같습니다. `X_train`, `X_test`는 모양이 맞는 float32이고 값이 0~1입니다. `y_train`의 앞 1,000장에 10종이 모두 있고 종류마다 그림 픽셀 합이 원래 데이터와 같습니다(그림과 정답의 짝이 유지됨). `y_test`는 종류 순서(종류마다 200장) 그대로입니다.

<details><summary>힌트</summary>

```text
접근방법:
- 섞는 순서 order 하나를 만들어 학습 그림과 학습 정답에 똑같이 씁니다.

세부구현:
1. randperm은 0부터 개수-1까지를 섞은 번호 텐서를 돌려줍니다. 첫 인자는 학습 그림 수, generator 인자는 제공 줄의 generator입니다.
2. 텐서를 order로 인덱싱하면 order의 번호 순서대로 꺼낸 새 텐서가 됩니다. 그림과 정답 모두 같은 order를 씁니다.
3. 정수 그림은 float로 바꾼 뒤 255로 나누면 0~1 실수가 됩니다. 채널 축은 unsqueeze로 1번 축 자리에 더합니다.
4. 그림과 정답을 서로 다른 순서로 섞으면 짝이 어긋나므로 randperm은 한 번만 부릅니다.
```

</details>

In [ ]:
# [작성] 학습 데이터를 섞고, 학습과 테스트 그림을 CNN 입력 모양으로 바꾸세요.

# 시드 0으로 고정한 generator(제공)와 섞는 순서
generator = torch.Generator().manual_seed(0)
order = ...

# 학습 데이터: order 순서로 꺼내고, 0~1 실수로 바꾸고, 채널 축을 더합니다
X_train = ...
y_train = ...

# 테스트 데이터: 섞지 않고 모양만 바꿉니다
X_test = ...
y_test = ...

print("X_train:", tuple(X_train.shape), X_train.dtype, "| y_train:", tuple(y_train.shape))
print("X_test:", tuple(X_test.shape), X_test.dtype, "| y_test:", tuple(y_test.shape))

In [ ]:
# [자가채점] 모양, 자료형, 값 범위, 섞기, 그림과 정답의 짝을 확인합니다.
for name, value in [("order", order), ("X_train", X_train), ("y_train", y_train), ("X_test", X_test), ("y_test", y_test)]:
    assert isinstance(value, torch.Tensor), f"{name}이 텐서가 아닙니다. `...`를 채웠는지 확인하세요."

# 모양과 자료형
assert X_train.shape == (10000, 1, 28, 28), f"X_train의 모양이 {tuple(X_train.shape)}입니다. (10000, 1, 28, 28)이 되도록 채널 축을 1번 축 자리에 더했는지 확인하세요."
assert X_test.shape == (2000, 1, 28, 28), f"X_test의 모양이 {tuple(X_test.shape)}입니다. 테스트 그림에도 같은 방법으로 채널 축을 더하세요."
assert X_train.dtype == torch.float32 and X_test.dtype == torch.float32, "X_train, X_test는 float32여야 합성곱 층에 넣을 수 있습니다. 정수 그림을 실수로 바꿨는지 확인하세요."
assert X_train.max().item() <= 1.0 and X_train.min().item() >= 0.0 and X_train.max().item() > 0.9, "그림 값은 0~1이어야 합니다. 255로 나눴는지 확인하세요."
assert X_test.max().item() <= 1.0 and X_test.max().item() > 0.9, "테스트 그림도 255로 나눠 0~1로 바꾸세요."
assert y_train.shape == (10000,) and y_train.dtype == torch.long, "y_train은 정수(torch.long) 정답 10,000개입니다. 학습 정답을 order 순서로 꺼냈는지 확인하세요."

# 섞기: 앞 1,000장에 10종이 모두 있어야 합니다
first_kinds = len(torch.unique(y_train[:1000]))
assert first_kinds == 10, f"y_train 앞 1,000장에 종류가 {first_kinds}개뿐입니다. 데이터가 종류 순서로 쌓여 있으니 order로 섞었는지 확인하세요."

# 그림과 정답의 짝: 종류마다 그림 픽셀 합이 원래 데이터와 같아야 합니다(그림과 정답을 다른 순서로 섞으면 여기서 걸립니다)
for c in range(10):
    original_sum = quickdraw["train_images"][quickdraw["train_labels"] == c].float().sum().item() / 255
    shuffled_sum = X_train[y_train == c].sum().item()
    assert abs(original_sum - shuffled_sum) < 1e-4 * original_sum, f"{class_names[c]} 그림과 정답의 짝이 어긋났습니다. 그림과 정답을 같은 order로 꺼냈는지 확인하세요."

# 섞는 순서: 0~9999가 한 번씩 들어 있고, 앞 20개가 시드 0 순서와 같아야 합니다
assert order.shape == (10000,) and torch.equal(torch.sort(order).values, torch.arange(10000)), "order는 0부터 9999까지를 한 번씩 섞은 번호 10,000개입니다. 번호의 개수가 학습 그림 수와 같은지 확인하세요."
seed0_order_head = [6044, 2890, 9399, 1917, 4919, 1729, 429, 7532, 337, 8223, 981, 4869, 784, 4840, 5592, 6301, 2920, 4053, 8506, 1426]
assert order[:20].tolist() == seed0_order_head, "order가 시드 0 순서와 다릅니다. 제공 줄의 generator를 섞는 함수에 넘겼는지, 그 함수를 한 번만 불렀는지 확인하세요."
seed0_label_head = [6, 2, 9, 1, 4, 1, 0, 7, 0, 8, 0, 4, 0, 4, 5, 6, 2, 4, 8, 1]
assert y_train[:20].tolist() == seed0_label_head, "y_train 앞 20개가 order 순서로 꺼낸 정답과 다릅니다. 학습 정답도 그림과 같은 order로 꺼내세요."

# 테스트 데이터는 섞지 않으므로 종류 순서(종류마다 200장) 그대로입니다
assert torch.equal(y_test, torch.arange(10).repeat_interleave(200)), "y_test가 종류 순서(0이 200개, 1이 200개, ...)가 아닙니다. 테스트 데이터는 섞지 않습니다."
for c in range(10):
    original_sum = quickdraw["test_images"][c * 200:(c + 1) * 200].float().sum().item() / 255
    test_sum = X_test[c * 200:(c + 1) * 200].sum().item()
    assert abs(original_sum - test_sum) < 1e-4 * original_sum, f"X_test의 {class_names[c]} 그림이 원래 테스트 그림과 다릅니다. 테스트 그림은 섞지 말고 0~1로만 바꾸세요."
print("✅ 1번 통과")

## 2. 미니배치를 나눠 주는 DataLoader 만들기

**배경**: 지난 시간처럼 64장씩 미니배치로 나눠 학습합니다. 학습용은 에폭마다 미니배치 순서를 섞고, 테스트용은 정확도만 계산하므로 섞지 않습니다.

**요구사항**
- (1) `X_train`, `y_train`을 묶어 미니배치 크기 64, 에폭마다 순서를 섞는 DataLoader **`train_loader`** 를 만듭니다.
- (2) `X_test`, `y_test`를 묶어 미니배치 크기 64, 순서를 섞지 않는 DataLoader **`test_loader`** 를 만듭니다.

**예시**: 학습 10,000장을 64장씩 나누면 미니배치 157개(마지막은 16장)이고, 한 에폭은 157스텝입니다.

**확인 기준**: 두 DataLoader가 데이터 장 수(10,000, 2,000)와 미니배치 크기 64를 가지고, `train_loader`만 섞습니다. `train_loader`의 첫 미니배치는 그림 `(64, 1, 28, 28)`, 정답 `(64,)`이고 `len(train_loader)`가 157입니다.

<details><summary>힌트</summary>

```text
접근방법:
- 그림과 정답을 이 순서로 한 쌍씩 묶는 데이터셋을 만들고, DataLoader에 미니배치 크기와 섞기 여부를 줍니다.

세부구현:
1. 그림과 정답을 한 쌍씩 묶을 때는 TensorDataset을 씁니다.
2. 섞기 여부는 shuffle 인자이고, 학습용만 켭니다.
```

</details>

In [ ]:
# [작성] 64장씩 나눠 주는 DataLoader를 만드세요.
from torch.utils.data import DataLoader, TensorDataset

# 학습용: 에폭마다 순서를 섞습니다
train_loader = ...

# 테스트용: 섞지 않습니다
test_loader = ...

print("학습 미니배치 수:", len(train_loader), "| 테스트 미니배치 수:", len(test_loader))

In [ ]:
# [자가채점] 미니배치 크기, 섞기 여부, 미니배치 모양을 확인합니다.
from torch.utils.data import RandomSampler, SequentialSampler

assert isinstance(train_loader, DataLoader) and isinstance(test_loader, DataLoader), "train_loader, test_loader는 DataLoader로 만드세요. `...`를 채웠는지 확인하세요."
assert len(train_loader.dataset) == 10000 and len(test_loader.dataset) == 2000, "train_loader에는 X_train, y_train을, test_loader에는 X_test, y_test를 묶어 넣습니다."
assert train_loader.batch_size == 64 and test_loader.batch_size == 64, "두 DataLoader 모두 미니배치 크기가 64입니다."
assert isinstance(train_loader.sampler, RandomSampler), "train_loader는 에폭마다 순서를 섞어야 합니다. 섞기 인자를 켰는지 확인하세요."
assert isinstance(test_loader.sampler, SequentialSampler), "test_loader는 섞지 않습니다. 섞기 인자를 껐는지 확인하세요."

# 첫 미니배치의 모양과 DataLoader의 길이
X_batch, y_batch = next(iter(train_loader))
assert X_batch.shape == (64, 1, 28, 28) and y_batch.shape == (64,), f"미니배치 하나는 그림 (64, 1, 28, 28), 정답 (64,)이어야 합니다. 지금은 {tuple(X_batch.shape)}, {tuple(y_batch.shape)}입니다. 그림, 정답 순서로 묶었는지 확인하세요."
assert len(train_loader) == 157, f"len(train_loader)가 {len(train_loader)}입니다. 10,000장을 64장씩 나누면 157개입니다."
print("✅ 2번 통과")

**결과 읽기**
- **157개:** 학습 10,000장을 64장씩 나눈 미니배치 수입니다(마지막은 16장).
- **32개:** 테스트 2,000장을 섞지 않고 64장씩 나눈 미니배치 수입니다.

## 3. CNN을 만들고 층마다 모양과 첫 손실 확인하기

**배경**: 5번에서 필터 수가 다른 CNN을 비교하므로, 필터 수 `n_filters`를 받아 CNN을 새로 만드는 함수를 만듭니다. 구조는 교안 01의 `make_cnn()`과 같고, 필터 수(첫 합성곱은 `n_filters`, 둘째 합성곱은 그 두 배)만 바꿀 수 있습니다. 출력층과 손실은 지난 시간의 다중 분류 짝(출력 10개와 `nn.CrossEntropyLoss`)입니다.

| 층 | 설정 | 출력 모양(`n_filters` = 16, 미니배치 64장) |
|---|---|---|
| 합성곱 1 | `nn.Conv2d(1, n_filters, kernel_size=3, padding=1)` + `nn.ReLU()` | (64, 16, 28, 28) |
| 풀링 1 | `nn.MaxPool2d(2)` | (64, 16, 14, 14) |
| 합성곱 2 | `nn.Conv2d(n_filters, n_filters * 2, kernel_size=3, padding=1)` + `nn.ReLU()` | (64, 32, 14, 14) |
| 풀링 2 | `nn.MaxPool2d(2)` | (64, 32, 7, 7) |
| 펼치기 | `nn.Flatten()` | (64, 1568) |
| 출력층 | `nn.Linear(n_filters * 2 * 7 * 7, 10)` | (64, 10) |

**요구사항**
- (1) 필터 수 `n_filters`를 받아 위 표의 CNN을 `nn.Sequential`로 만들어 계산 장치(`device`)에 올려 돌려주는 함수 **`make_filter_cnn(n_filters)`** 를 만듭니다.
    - 모델을 만들기 직전에 시드 0을 고정해, 언제 불러도 같은 초깃값으로 시작하게 합니다.
- (2) `make_filter_cnn(16)`을 담은 **`model`** 을 만듭니다.
- (3) 학습 미니배치 `X_batch`(제공 줄)를 `model`의 층마다 차례로 통과시키며, `층`(층 종류 이름), `출력 모양` 두 키의 dict를 모은 리스트 **`shape_rows`** 를 만듭니다.
- (4) `nn.CrossEntropyLoss()`를 담은 **`loss_fn`** 을 만듭니다.
- (5) 학습 전 `model`에 학습 데이터 앞 1,000장(`X_train[:1000]`, `y_train[:1000]`)을 넣어 계산한 손실을 파이썬 실수 **`first_loss`** 로 만듭니다.

**예시**: `n_filters`가 8이면 두 번째 합성곱의 필터는 16개, 펼친 크기는 16 x 7 x 7 = 784입니다.

**확인 기준**: `make_filter_cnn(8)`이 위 표대로 8층이고 두 번 불러도 첫 층 가중치가 같습니다. `shape_rows`는 8개이고 층마다 출력 모양이 표와 같으며 마지막이 `(64, 10)`입니다. `loss_fn`은 `nn.CrossEntropyLoss`이고, `first_loss`는 log 10(2.30)과 0.1 이내이면서 학습 전 `model`로 학습 데이터 앞 1,000장을 평가한 손실과 같습니다.

<details><summary>힌트</summary>

```text
접근방법:
- 교안 01의 CNN에서 필터 수 16과 32 자리에 n_filters와 n_filters의 2배를 넣습니다.

세부구현:
1. nn.Sequential은 for 문으로 층을 하나씩 꺼낼 수 있습니다. 층을 함수처럼 불러 텐서를 넣고, 그 결과를 같은 변수에 다시 담아 다음 층으로 넘깁니다.
2. 층의 종류 이름은 type과 __name__으로 얻습니다.
3. mat1 and mat2 shapes cannot be multiplied 오류가 나면 Linear의 입력 크기가 Flatten 뒤 크기(채널 x 7 x 7)와 다른 것입니다.
4. 입력은 모델과 같은 장치에 있어야 하므로 학습 그림과 정답을 계산 장치로 옮깁니다.
5. 손실 텐서에서 파이썬 실수를 꺼낼 때는 item을 씁니다.
```

</details>

In [ ]:
# [작성] (1) 필터 수를 받아 CNN을 새로 만드는 함수를 완성하세요.
import torch.nn as nn


def make_filter_cnn(n_filters):
    """첫 합성곱 필터 n_filters개, 두 번째 합성곱 필터 n_filters의 2배인 CNN을 새로 만들어 돌려줍니다."""
    ...


print(make_filter_cnn(16))

In [ ]:
# [작성] (2), (3) 모델을 만들고, 학습 미니배치 하나를 층마다 통과시키며 출력 모양을 모으세요.
model = ...

# 학습 미니배치 하나(제공)
X_batch, y_batch = next(iter(train_loader))

shape_rows = []
x = X_batch.to(device)
for layer in model:
    # 이 층을 통과시키고 층 이름과 출력 모양을 기록합니다
    ...

shape_table = pd.DataFrame(shape_rows)
display(shape_table)

In [ ]:
# [작성] (4), (5) 손실함수를 준비하고 학습 전 첫 손실을 계산하세요.
import math

loss_fn = ...

# 학습 데이터 앞 1,000장으로 계산한 학습 전 손실
first_loss = ...
print("첫 손실:", round(first_loss, 4), "| log 10:", round(math.log(10), 4))

In [ ]:
# [자가채점] 층 구성, 같은 초깃값, 층마다 모양, 첫 손실을 확인합니다.
model_a = make_filter_cnn(8)
assert isinstance(model_a, nn.Sequential), "make_filter_cnn()은 nn.Sequential 모델을 돌려줘야 합니다. 함수 끝에서 모델을 돌려주는지 확인하세요."
assert len(model_a) == 8, f"층은 Conv2d, ReLU, MaxPool2d, Conv2d, ReLU, MaxPool2d, Flatten, Linear 8개입니다. 지금은 {len(model_a)}개입니다."

# 층마다 종류와 크기(n_filters = 8로 확인)
assert isinstance(model_a[0], nn.Conv2d) and model_a[0].in_channels == 1 and model_a[0].out_channels == 8, "첫 층은 입력 채널 1, 출력 채널 n_filters인 nn.Conv2d입니다. 필터 수 자리에 함수의 n_filters를 넣었는지 확인하세요."
assert model_a[0].kernel_size == (3, 3) and model_a[0].padding == (1, 1), "합성곱 층은 kernel_size 3, padding 1입니다. padding이 1이어야 28x28 크기가 유지됩니다."
assert isinstance(model_a[1], nn.ReLU) and isinstance(model_a[4], nn.ReLU), "합성곱 층 바로 뒤에 ReLU를 넣으세요."
assert isinstance(model_a[2], nn.MaxPool2d) and isinstance(model_a[5], nn.MaxPool2d), "ReLU 뒤에 크기 2인 MaxPool2d를 넣으세요."
assert isinstance(model_a[3], nn.Conv2d) and model_a[3].in_channels == 8 and model_a[3].out_channels == 16, "두 번째 합성곱은 입력 채널 n_filters, 출력 채널 n_filters의 2배입니다."
assert isinstance(model_a[6], nn.Flatten), "Linear 앞에 Flatten으로 (채널, 높이, 너비)를 한 줄로 펼치세요."
assert isinstance(model_a[7], nn.Linear) and model_a[7].in_features == 16 * 7 * 7 and model_a[7].out_features == 10, "출력층은 입력이 펼친 크기(n_filters의 2배 x 7 x 7), 출력이 낙서 종류 수 10인 nn.Linear입니다."
assert next(model_a.parameters()).device.type == device.type, "모델을 계산 장치에 올렸는지 확인하세요."

# 부를 때마다 같은 초깃값
model_b = make_filter_cnn(8)
assert torch.equal(model_a[0].weight, model_b[0].weight), "make_filter_cnn()을 두 번 부르면 첫 층 가중치가 같아야 합니다. 모델을 만들기 직전에 시드 0을 고정했는지 확인하세요."

# 층마다 출력 모양
assert isinstance(model, nn.Sequential) and model[0].out_channels == 16, "model에는 필터 수 16으로 만든 모델을 담으세요."
assert isinstance(shape_rows, list) and len(shape_rows) == 8, "shape_rows에는 층마다 하나씩, 8개가 들어가야 합니다. 반복문 안에서 더했는지 확인하세요."
assert list(shape_rows[0]) == ["층", "출력 모양"], "shape_rows의 dict는 층, 출력 모양 두 키를 이 순서로 가집니다."
assert shape_rows[0]["층"] == "Conv2d" and shape_rows[7]["층"] == "Linear", "층에는 층의 종류 이름(Conv2d, ReLU, ...)을 담으세요."
expected_shapes = [(64, 16, 28, 28), (64, 16, 28, 28), (64, 16, 14, 14), (64, 32, 14, 14), (64, 32, 14, 14), (64, 32, 7, 7), (64, 1568), (64, 10)]
for row, expected in zip(shape_rows, expected_shapes):
    assert tuple(row["출력 모양"]) == expected, f"{row['층']} 층의 출력 모양이 {row['출력 모양']}입니다. {expected}여야 합니다. 앞 층의 출력을 다음 층에 넘겼는지 확인하세요."

# 첫 손실: log 10 근처이면서 학습 데이터 앞 1,000장으로 계산한 값
assert isinstance(loss_fn, nn.CrossEntropyLoss), "loss_fn은 nn.CrossEntropyLoss입니다."
assert isinstance(first_loss, float), "first_loss는 파이썬 실수입니다. 손실 텐서에서 숫자를 꺼냈는지 확인하세요."
assert abs(first_loss - math.log(10)) < 0.1, f"첫 손실 {first_loss:.4f}가 log 10(2.303)에서 멉니다. 학습 전 모델의 손실인지 확인하세요."
first_loss_check, _ = evaluate(model, DataLoader(TensorDataset(X_train[:1000], y_train[:1000]), batch_size=64), loss_fn)
assert abs(first_loss - first_loss_check) < 0.0005, f"first_loss({first_loss:.4f})가 학습 전 model로 X_train, y_train의 앞 1,000장을 평가한 손실({first_loss_check:.4f})과 다릅니다. 테스트 데이터가 아니라 학습 데이터 앞 1,000장을 넣었는지 확인하세요."
print("✅ 3번 통과")

**결과 읽기**
- **(64, 1568)에서 (64, 10):** Flatten이 32 x 7 x 7 = 1,568개를 한 줄로 펼치고, Linear가 종류 수 10개의 로짓으로 줄입니다.
- **첫 손실 2.299:** log 10(2.303)과 거의 같습니다. 학습 전 모델은 무작위로 고를 때와 같은 수준입니다.

## 4. 학습 손실과 학습 데이터 정확도를 기록하는 학습 함수 만들기

**배경**: 이 뒤로 필터 수를 바꿔 새 CNN을 정해진 에폭만큼 학습하는 일이 여러 번 나오므로 함수 하나로 만듭니다. 테스트 데이터는 6번에서 최종 모델을 학습한 뒤 한 번만 보므로, 여기서는 학습 데이터만 씁니다.

**요구사항**
- (1) 에폭 수 **`num_epochs`** 를 5로 둡니다.
- (2) 필터 수와 DataLoader를 받아 `(학습한 모델, 에폭별 기록 표)`를 돌려주는 함수 **`train_cnn(n_filters, loader)`** 를 만듭니다.
    - 함수 안에서 `make_filter_cnn(n_filters)`로 새 모델을, 그 모델의 파라미터로 학습률 0.001인 Adam 옵티마이저를 만듭니다.
    - `num_epochs`번 반복하며 `train_one_epoch(모델, loader, loss_fn, 옵티마이저)`로 학습 손실을 구합니다.
    - 같은 에폭이 끝난 모델의 학습 데이터 정확도는 `evaluate(모델, eval_loader, loss_fn)`으로 구합니다. `eval_loader`는 섞지 않는 DataLoader로, 제공 줄에 있습니다.
    - 표의 열은 `에폭`(1부터), `학습 손실`, `학습 데이터 정확도`입니다.
- (3) `train_cnn(16, train_loader)`가 돌려준 모델을 담은 **`model`** 을 만듭니다(3번의 `model`을 대체합니다).
- (4) 같은 호출이 돌려준 기록 표 **`history_table`**(5줄)을 만듭니다.

**예시**: `train_cnn(16, train_loader)`가 돌려주는 표의 한 줄은 `{"에폭": 1, "학습 손실": ..., "학습 데이터 정확도": ...}`입니다.

**확인 기준**: `history_table`이 5줄이고 학습 손실이 줄며 마지막 학습 데이터 정확도가 0.85 이상입니다. 같은 인자로 `train_cnn`을 다시 불러도 첫 에폭 손실이 거의 같고, 128장짜리 DataLoader를 넘기면 기록된 학습 데이터 정확도가 그 DataLoader의 데이터로 다시 계산한 정확도와 거의 같습니다.

<details><summary>힌트</summary>

```text
접근방법:
- 새 모델, 옵티마이저, 에폭 반복(학습 한 번, 정확도 계산 한 번, 기록 한 줄) 순서로 쓰고 모델과 표를 함께 돌려줍니다.

세부구현:
1. evaluate는 (평균 손실, 정확도) 두 값을 돌려줍니다. 정확도만 쓸 때는 손실 자리를 밑줄 변수로 받습니다.
2. 에폭 번호는 range가 주는 번호에 1을 더합니다.
3. 옵티마이저에는 함수 안에서 만든 모델의 parameters를 넘깁니다. 다른 모델의 파라미터를 넘기면 학습 데이터 정확도가 0.1 근처에 머뭅니다.
4. 학습은 함수가 받은 loader로 합니다. 함수 밖의 train_loader를 쓰면 5번에서 앞 3,000장이 아니라 10,000장으로 학습됩니다.
```

</details>

In [ ]:
# [작성] (1), (2) 새 CNN을 num_epochs만큼 학습하며 학습 손실과 학습 데이터 정확도를 기록하는 함수를 완성하세요.
num_epochs = 5


def train_cnn(n_filters, loader):
    """make_filter_cnn(n_filters)로 새 모델을 만들어 loader로 num_epochs 에폭 학습하고, (모델, 에폭별 기록 표)를 돌려줍니다."""
    # 학습 데이터 정확도를 계산할 DataLoader(제공): 같은 데이터를 섞지 않고 읽습니다
    eval_loader = DataLoader(loader.dataset, batch_size=64, shuffle=False)

    # 새 모델과 옵티마이저
    ...

    # 에폭마다 학습하고, 학습 손실과 학습 데이터 정확도를 기록합니다
    history = []
    ...

    return ...

In [ ]:
# [작성] (3), (4) 필터 16개인 CNN을 학습 데이터 전체로 학습하세요.
model, history_table = ...
display(history_table.set_index("에폭").round(3))

In [ ]:
# [자가채점] 함수가 매번 새 모델에서 시작하는지, 학습이 실제로 됐는지 확인합니다.
assert num_epochs == 5, "num_epochs는 5입니다."
assert isinstance(model, nn.Sequential) and model[0].out_channels == 16, "model에는 필터 수 16으로 학습한 모델을 담으세요. 함수가 모델, 표 순서로 돌려주는지 확인하세요."
assert isinstance(history_table, pd.DataFrame), "train_cnn은 기록 리스트를 DataFrame으로 만들어 돌려줍니다."
assert list(history_table.columns) == ["에폭", "학습 손실", "학습 데이터 정확도"], f"기록 표의 열은 에폭, 학습 손실, 학습 데이터 정확도입니다. 지금은 {list(history_table.columns)}입니다."
assert len(history_table) == num_epochs and history_table["에폭"].tolist() == [1, 2, 3, 4, 5], "기록 표는 에폭마다 한 줄, 에폭 번호는 1부터 5까지입니다."

# 학습이 됐는지
losses = history_table["학습 손실"].tolist()
last_acc = history_table["학습 데이터 정확도"].iloc[-1]
assert losses[-1] < losses[0], "학습 손실이 줄지 않았습니다. 에폭마다 제공 함수 train_one_epoch로 loader를 학습했는지 확인하세요."
assert last_acc >= 0.85, f"마지막 학습 데이터 정확도 {last_acc:.3f}가 낮습니다. 옵티마이저에 함수 안에서 만든 모델의 파라미터를 넘겼는지 확인하세요."

# 학습 데이터 정확도는 학습 데이터에서 계산합니다(테스트 데이터로 계산하면 여기서 걸립니다)
check_train_loss, check_train_acc = evaluate(model, DataLoader(train_loader.dataset, batch_size=64), loss_fn)
assert abs(last_acc - check_train_acc) < 0.002, "학습 데이터 정확도가 학습한 model로 train_loader의 데이터를 다시 계산한 정확도와 다릅니다. 제공 줄의 eval_loader로 계산했는지 확인하세요. 테스트 데이터는 6번에서만 씁니다."

# 함수가 n_filters를 쓰고, 매번 새 모델에서 시작하는지 작은 데이터로 확인합니다
tiny_loader = DataLoader(TensorDataset(X_train[:128], y_train[:128]), batch_size=64, shuffle=True)
tiny_model, tiny_history_1 = train_cnn(8, tiny_loader)
assert tiny_model[0].out_channels == 8, "train_cnn(8, ...)의 모델은 필터 8개여야 합니다. 함수 안에서 받은 n_filters로 모델을 만들었는지 확인하세요."

# 함수가 받은 loader를 쓰는지: 128장 DataLoader의 데이터로 정확도를 다시 계산해 기록과 비교합니다
tiny_loss, tiny_acc = evaluate(tiny_model, DataLoader(tiny_loader.dataset, batch_size=64), loss_fn)
tiny_last_acc = tiny_history_1["학습 데이터 정확도"].iloc[-1]
assert abs(tiny_last_acc - tiny_acc) < 0.005, f"train_cnn에 128장짜리 DataLoader를 넘겼는데, 기록된 마지막 학습 데이터 정확도({tiny_last_acc:.4f})가 그 데이터로 다시 계산한 정확도({tiny_acc:.4f})와 다릅니다. 함수 안에서 밖의 train_loader가 아니라 인자로 받은 loader로 학습하고, 제공 줄의 eval_loader로 평가하세요."
tiny_first_loss = tiny_history_1["학습 손실"].iloc[0]
assert tiny_first_loss > 2.0, f"128장(한 에폭에 2스텝)으로 학습하면 첫 에폭 학습 손실이 log 10(2.30) 근처여야 하는데 {tiny_first_loss:.4f}입니다. train_one_epoch에도 인자로 받은 loader를 넘기세요."

# 다시 불러도 같은 첫 에폭 손실이 나와야 합니다
tiny_model_2, tiny_history_2 = train_cnn(8, tiny_loader)
assert abs(tiny_history_1["학습 손실"].iloc[0] - tiny_history_2["학습 손실"].iloc[0]) < 0.01, "train_cnn을 다시 불렀더니 첫 에폭 손실이 달라졌습니다. 함수 안에서 새 모델과 그 모델의 새 옵티마이저를 만들었는지 확인하세요."
print("✅ 4번 통과")

**결과 읽기**
- **0.980에서 0.284:** 학습 손실이 1에폭부터 5에폭까지 줄어듭니다.
- **0.826에서 0.926:** 학습 데이터 정확도가 무작위로 고를 때의 0.1보다 훨씬 높게 시작해 에폭마다 오릅니다.

## 5. 필터 수 후보가 학습 데이터에 맞추는 정도 비교하기

**배경**: 첫 합성곱의 필터 수(그림에서 찾는 특징의 가짓수)를 8, 16, 32로 바꿔 학습하고, 필터가 많을수록 학습 데이터에 얼마나 더 잘 맞추는지 비교합니다. 후보마다 처음부터 학습해야 하므로 학습 데이터 앞 3,000장으로 빠르게 비교합니다.

학습 손실은 파라미터가 많은 모델이 대체로 더 낮게 나오므로 필터 수를 고르는 기준이 될 수 없습니다. 검증 데이터로 고르는 방법은 교안 02에서 배우고, 이 과제는 6번에서 필터 수를 32로 정해 두고 학습합니다.

**요구사항**
- (1) 1번에서 섞어 둔 `X_train`, `y_train`의 앞 3,000장으로 미니배치 64, 에폭마다 순서를 섞는 DataLoader **`subset_loader`** 를 만듭니다.
- (2) `filter_candidates`(제공 줄)의 필터 수마다 `train_cnn(필터 수, subset_loader)`로 학습한 모델을 **`candidate_model`** 에 담습니다. 반복문이 끝나면 마지막 후보(필터 32개)의 모델이 남습니다.
- (3) 같은 호출이 돌려준 기록 표를 **`candidate_history`** 에 담습니다(10번 비교에 쓰려고 `filter_curves`에 보관하는 줄은 제공).
- (4) 후보마다 `필터 수`, `마지막 학습 손실`, `마지막 학습 데이터 정확도` 세 키의 dict를 이 순서로 더한 리스트 **`compare_rows`**(3개)를 만듭니다.

**예시**: 1번에서 섞었으므로 `subset_loader`의 3,000장에도 10종이 고르게 들어 있습니다.

**확인 기준**: `subset_loader`는 3,000장이고 `X_train`의 앞 3,000장에서 만들었습니다. 후보 3개가 `filter_candidates` 순서대로 있고, 표에 테스트 데이터로 계산한 열이 없으며, 필터가 많을수록 마지막 학습 손실이 작습니다. `candidate_model`로 `subset_loader`의 데이터 정확도를 다시 계산하면 `compare_rows` 마지막 줄의 마지막 학습 데이터 정확도와 거의 같습니다.

<details><summary>힌트</summary>

```text
접근방법:
- 반복문 한 번에 train_cnn 한 번, compare_rows에 한 줄을 더합니다.

세부구현:
1. 앞 3,000장은 X_train과 y_train을 3,000까지 슬라이싱한 것입니다. 2번과 같은 방법으로 묶어 DataLoader를 만듭니다.
2. 기록 표의 마지막 줄 값은 열을 고른 뒤 iloc에 -1을 주어 꺼냅니다.
3. train_cnn에 train_loader를 넘기면 앞 3,000장이 아니라 10,000장으로 학습합니다.
```

</details>

In [ ]:
# [작성] (1) 학습 데이터 앞 3,000장으로 DataLoader를 만드세요.
subset_loader = ...
print(f"비교용 학습 그림: {len(subset_loader.dataset):,}장 | 미니배치 수: {len(subset_loader)}")

In [ ]:
# [작성] (2), (3), (4) 필터 수 후보마다 학습해 마지막 학습 손실과 학습 데이터 정확도를 모으세요.
filter_candidates = [8, 16, 32]

compare_rows = []
filter_curves = {}
for n_filters in filter_candidates:
    # 필터 수 n_filters로 새 CNN 학습
    candidate_model, candidate_history = ...

    # 에폭별 기록은 10번 비교에 쓰려고 보관합니다(제공)
    filter_curves[n_filters] = candidate_history

    # 필터 수, 마지막 학습 손실, 마지막 학습 데이터 정확도
    compare_rows.append(...)

compare_table = pd.DataFrame(compare_rows)
display(compare_table.round(3))

In [ ]:
# [자가채점] 비교용 데이터와 후보별 기록을 확인합니다.
assert isinstance(subset_loader, DataLoader) and len(subset_loader.dataset) == 3000, "subset_loader에는 X_train, y_train의 앞 3,000장을 묶어 넣습니다."
assert torch.equal(subset_loader.dataset[0][0], X_train[0]), "subset_loader는 1번에서 섞은 X_train의 앞 3,000장으로 만드세요."
assert subset_loader.batch_size == 64 and isinstance(subset_loader.sampler, RandomSampler), "subset_loader도 미니배치 크기 64, 순서 섞기입니다."

assert filter_candidates == [8, 16, 32], "필터 수 후보 목록(filter_candidates)은 바꾸지 마세요."
assert isinstance(compare_rows, list) and len(compare_rows) == 3, "후보마다 한 줄, 3줄을 compare_rows에 더하세요. `...`를 채웠는지 확인하세요."
for row, n_filters in zip(compare_rows, filter_candidates):
    assert list(row) == ["필터 수", "마지막 학습 손실", "마지막 학습 데이터 정확도"], "compare_rows의 dict는 필터 수, 마지막 학습 손실, 마지막 학습 데이터 정확도 세 키만 이 순서로 가집니다. 테스트 데이터로 계산한 값은 넣지 않습니다."
    assert row["필터 수"] == n_filters, "compare_rows는 filter_candidates 순서대로 만드세요."
    curve = filter_curves[n_filters]
    assert abs(row["마지막 학습 손실"] - curve["학습 손실"].iloc[-1]) < 1e-6, f"필터 {n_filters}개의 마지막 학습 손실은 train_cnn이 돌려준 표의 마지막 줄 값입니다."
    assert abs(row["마지막 학습 데이터 정확도"] - curve["학습 데이터 정확도"].iloc[-1]) < 1e-9, f"필터 {n_filters}개의 마지막 학습 데이터 정확도는 train_cnn이 돌려준 표의 마지막 줄 값입니다."

# subset_loader로 학습했는지: 마지막 후보(필터 32개) 모델의 앞 3,000장 정확도를 다시 계산해 기록과 비교합니다
assert isinstance(candidate_model, nn.Sequential) and candidate_model[0].out_channels == 32, "candidate_model에는 train_cnn이 돌려준 모델을 담으세요. 반복문이 끝나면 마지막 후보(필터 32개)의 모델이 남습니다."
check_subset_loss, check_subset_acc = evaluate(candidate_model, DataLoader(subset_loader.dataset, batch_size=64), loss_fn)
recorded_acc = compare_rows[-1]["마지막 학습 데이터 정확도"]
assert abs(recorded_acc - check_subset_acc) < 0.005, f"필터 32개의 마지막 학습 데이터 정확도({recorded_acc:.4f})가 subset_loader의 데이터로 다시 계산한 정확도({check_subset_acc:.4f})와 다릅니다. train_cnn에 train_loader가 아니라 subset_loader를 넘기세요."

# 필터가 많을수록 학습 데이터에는 더 잘 맞습니다
last_losses = compare_table["마지막 학습 손실"].tolist()
assert last_losses[0] > last_losses[1] > last_losses[2], "필터 8, 16, 32 순서로 마지막 학습 손실이 작아져야 합니다. train_cnn에 반복문의 n_filters와 subset_loader를 넘겼는지 확인하세요."
print("✅ 5번 통과")

**결과 읽기**
- **0.586, 0.442, 0.314:** 필터 8, 16, 32개 순으로 마지막 학습 손실이 작아집니다.
- **0.838, 0.871, 0.930:** 필터가 많을수록 앞 3,000장에 더 잘 맞춥니다. 학습 데이터에 대한 값이라 필터 수를 고르는 근거로 쓰지 않습니다.

## 6. 필터 32개로 최종 모델을 학습하고 테스트 정확도 계산하기

**배경**: 필터 수를 정한 뒤에 학습 데이터 10,000장 전체로 최종 모델을 학습하고, 학습에 쓰지 않은 테스트 데이터로 정확도를 **한 번** 계산합니다. 학습 데이터 정확도와 테스트 정확도를 나란히 놓으면, 처음 보는 낙서에서 정확도가 얼마나 떨어지는지 알 수 있습니다.

**요구사항**
- (1) 최종 모델의 필터 수 **`final_filters`** 는 32입니다(제공 줄).
- (2) `train_cnn(final_filters, train_loader)`가 돌려준 모델 **`final_model`** 을 만듭니다.
- (3) 같은 호출이 돌려준 기록 표 **`final_history`** 를 만듭니다.
- (4) `evaluate`로 `final_model`의 `test_loader` 손실을 파이썬 실수 **`test_loss`** 로 만듭니다.
- (5) 같은 호출의 정확도를 파이썬 실수 **`test_acc`** 로 만듭니다.

**확인 기준**: `final_model`의 첫 합성곱 필터 수가 32이고, `final_history`의 학습 데이터 정확도가 10,000장 기준 비율이며, `test_acc`가 테스트 2,000장 기준 비율로 0.85 이상이고 마지막 학습 데이터 정확도와 다릅니다.

<details><summary>힌트</summary>

```text
접근방법:
- 학습 한 번, 평가 한 번입니다.

세부구현:
1. 이번에는 subset_loader가 아니라 학습 데이터 전체인 train_loader로 학습합니다.
2. evaluate에 모델, 테스트용 DataLoader, 손실함수를 넘기면 (평균 손실, 정확도) 두 값을 돌려줍니다.
```

</details>

In [ ]:
# [작성] 필터 32개로 최종 모델을 학습하고 테스트 데이터로 한 번 평가하세요.

# 최종 모델의 필터 수(제공)
final_filters = 32

# 학습 데이터 전체로 최종 모델 학습
final_model, final_history = ...

# 테스트 데이터로 한 번 평가
test_loss, test_acc = ...
print("마지막 학습 데이터 정확도:", round(final_history["학습 데이터 정확도"].iloc[-1], 3), "| 테스트 정확도:", round(test_acc, 3))

In [ ]:
# [자가채점] 최종 모델과 테스트 정확도를 확인합니다.
assert final_filters == 32, "final_filters는 32입니다(제공 줄)."
assert isinstance(final_model, nn.Sequential) and final_model[0].out_channels == final_filters, "final_model은 train_cnn이 돌려준 모델입니다. 필터 수 자리에 final_filters를 넘겼는지 확인하세요."
assert isinstance(final_history, pd.DataFrame) and len(final_history) == num_epochs, "final_history는 train_cnn이 돌려준 기록 표입니다."

# 학습 데이터 전체로 학습했는지: 10,000장에서 맞힌 비율이면 10,000을 곱했을 때 정수입니다(3,000장으로 학습하면 대개 여기서 걸립니다)
for train_acc in final_history["학습 데이터 정확도"].tolist():
    assert abs(train_acc * 10000 - round(train_acc * 10000)) < 1e-6, "final_history의 학습 데이터 정확도가 10,000장 기준 비율이 아닙니다. 학습 데이터 10,000장만으로 학습했는지(subset_loader나 테스트 데이터를 섞지 않았는지) 확인하세요."

# 테스트 정확도: 테스트 2,000장에서 맞힌 비율이고, 학습 데이터 정확도와는 다른 값입니다
assert isinstance(test_loss, float) and isinstance(test_acc, float), "test_loss, test_acc는 evaluate가 돌려준 두 값(평균 손실, 정확도)을 차례로 받은 파이썬 실수입니다."
assert 0.0 <= test_acc <= 1.0 and abs(test_acc * 2000 - round(test_acc * 2000)) < 1e-6, "test_acc가 테스트 2,000장에서 맞힌 비율이 아닙니다. evaluate에 테스트용 DataLoader를 넣었는지 확인하세요."
assert abs(test_acc - final_history["학습 데이터 정확도"].iloc[-1]) > 1e-9, "test_acc가 마지막 학습 데이터 정확도와 같습니다. 학습 데이터가 아니라 테스트 데이터로 평가하세요."
assert test_acc >= 0.85, f"테스트 정확도 {test_acc:.3f}가 낮습니다. 학습 데이터 전체를 학습한 final_model로 평가했는지 확인하세요."
print("✅ 6번 통과")

**결과 읽기**
- **학습 데이터 정확도 0.945, 테스트 정확도 0.912:** 처음 보는 낙서에서는 0.033 낮습니다. 학습에 쓴 낙서에 조금 더 맞춰져 있다는 뜻입니다.
- **소수 셋째 자리:** 계산 장치와 스레드 수에 따라 달라질 수 있습니다. 결론은 두 정확도의 차이로 읽습니다.

## 7. 틀린 낙서를 보고 가장 많이 헷갈린 짝 찾기

**배경**: 정확도 숫자 하나로는 어떤 종류를 어떤 종류로 착각하는지 알 수 없으므로, 테스트 낙서마다 (정답, 예측) 짝을 세어 10 x 10 혼동 행렬(confusion matrix)로 만듭니다. 대각선(정답 = 예측)은 맞힌 수이고, 대각선 밖에서 가장 큰 칸이 가장 많이 헷갈린 짝입니다.

**요구사항**
- (1) `final_model`로 `X_test` 전체의 예측 번호(로짓이 가장 큰 종류)를 구한 CPU 텐서 **`test_predictions`**(정수 2,000개)를 만듭니다. 학습이 아니므로 `torch.no_grad()` 안에서 계산합니다.
- (2) 0으로 채운 `(10, 10)` 정수 텐서 **`confusion`** 을 만들고, 테스트 낙서마다 `confusion[정답, 예측]`에 1을 더합니다.
- (3) 대각선을 뺀 칸 가운데 가장 큰 칸의 정답 번호 **`top_true`**(파이썬 정수)를 만듭니다.
- (4) 같은 칸의 예측 번호 **`top_pred`**(파이썬 정수)를 만듭니다.
- (5) 같은 칸의 개수 **`top_count`**(파이썬 정수)를 만듭니다.

**예시**: 표의 (고양이 행, 물고기 열) 칸이 5라면, 정답이 고양이인 낙서 5장을 물고기로 예측한 것입니다.

**확인 기준**: `test_predictions`는 0~9 정수 2,000개이고 그 정확도가 `test_acc`와 거의 같습니다(0.003 이내). `confusion`의 합이 2,000이고 행마다 합이 200이며 대각선의 합을 2,000으로 나눈 값이 `test_acc`와 거의 같습니다. `top_true`와 `top_pred`가 다르고 `top_count`가 대각선 밖에서 가장 큰 값입니다.

<details><summary>힌트</summary>

```text
접근방법:
- 예측 번호를 구하고, (정답, 예측) 짝마다 한 칸에 1을 더한 뒤, 대각선 밖 칸을 하나씩 보며 가장 큰 칸을 찾습니다.

세부구현:
1. 로짓이 가장 큰 번호는 argmax로 찾습니다. dim에는 종류별 로짓이 놓인 축 번호를 주고, 결과는 cpu로 CPU에 옮깁니다.
2. 빈 표는 torch.zeros로 만듭니다. 모양은 10행 10열, dtype은 정수(torch.long)입니다.
3. 정답과 예측을 tolist로 파이썬 리스트로 바꾼 뒤 zip으로 묶으면 한 쌍씩 꺼낼 수 있습니다.
4. 가장 큰 칸은 정답 번호와 예측 번호를 0부터 9까지 이중 반복문으로 돌며 찾고, 두 번호가 같은 칸은 건너뜁니다.
5. 표의 칸 값은 텐서이므로 item으로 파이썬 정수를 꺼냅니다.
```

</details>

In [ ]:
# [작성] (1) 최종 모델로 테스트 낙서 2,000장의 예측 번호를 구하세요.
with torch.no_grad():
    test_predictions = ...

print("예측 개수:", len(test_predictions), "| 앞 10개:", test_predictions[:10].tolist())

In [ ]:
# [작성] (2) 정답과 예측의 짝을 세어 10 x 10 표를 만드세요.
confusion = ...
...

# 행은 정답, 열은 예측
confusion_table = pd.DataFrame(confusion.numpy(), index=class_names, columns=class_names)
confusion_table.index.name = "정답"
confusion_table.columns.name = "예측"
display(confusion_table)

In [ ]:
# [작성] (3), (4), (5) 대각선 밖에서 가장 큰 칸을 찾으세요.
top_true = 0
top_pred = 0
top_count = 0
...

print(f"가장 많이 헷갈린 짝: 정답 {class_names[top_true]} -> 예측 {class_names[top_pred]}, {top_count}장")

In [ ]:
# [자가채점] 예측, 짝 세기 표, 가장 많이 헷갈린 짝을 확인합니다.
assert isinstance(test_predictions, torch.Tensor) and test_predictions.shape == (2000,), "test_predictions는 테스트 낙서마다 예측 번호 하나, 모양 (2000,)인 텐서입니다. argmax의 dim에 종류별 로짓이 놓인 축을 주었는지 확인하세요."
assert test_predictions.device.type == "cpu", "test_predictions는 CPU 텐서여야 합니다."
assert test_predictions.dtype == torch.long and test_predictions.min().item() >= 0 and test_predictions.max().item() <= 9, "test_predictions는 종류 번호(0~9) 정수입니다. 로짓 자체가 아니라 로짓이 가장 큰 번호를 담으세요."

# 예측으로 계산한 정확도가 6번의 test_acc와 같아야 합니다(final_model이 아닌 모델로 예측하면 여기서 걸립니다)
predictions_acc = torch.eq(test_predictions, y_test).double().mean().item()
assert abs(predictions_acc - test_acc) < 0.003, f"test_predictions로 계산한 정확도({predictions_acc:.4f})가 test_acc({test_acc:.4f})와 다릅니다. test_acc와 test_predictions를 모두 final_model로 계산했는지 확인하세요."

# 짝 세기 표
assert isinstance(confusion, torch.Tensor) and confusion.shape == (10, 10), "confusion은 모양 (10, 10)인 정수 텐서입니다. 0으로 채운 표에서 시작하세요."
assert confusion.sum().item() == 2000, f"confusion의 합이 {confusion.sum().item()}입니다. 테스트 낙서 2,000장마다 한 칸에 1씩 더하세요."
assert abs(confusion.diag().sum().item() / 2000 - test_acc) < 0.003, "confusion의 대각선 합이 맞힌 수와 다릅니다. 행은 정답, 열은 예측인 칸에 더했는지 확인하세요."
assert confusion.sum(dim=1).tolist() == [200] * 10, "confusion의 행마다 합은 그 종류의 테스트 낙서 수 200입니다. 행에 정답, 열에 예측을 넣었는지 확인하세요."

# 가장 많이 헷갈린 짝
off_diagonal = confusion.clone()
off_diagonal.fill_diagonal_(0)
assert isinstance(top_true, int) and isinstance(top_pred, int) and isinstance(top_count, int), "top_true, top_pred, top_count는 파이썬 정수입니다. 표의 칸 값에서 숫자를 꺼냈는지 확인하세요."
assert top_true != top_pred, "top_true와 top_pred가 같습니다. 대각선(정답 = 예측) 칸은 건너뛰세요."
assert top_count == off_diagonal.max().item(), f"top_count({top_count})가 대각선 밖에서 가장 큰 값({off_diagonal.max().item()})이 아닙니다."
assert confusion[top_true, top_pred].item() == top_count, "top_true, top_pred 칸의 값이 top_count와 다릅니다. 가장 큰 칸을 찾을 때 세 값을 함께 바꾸세요."
print("✅ 7번 통과")

In [ ]:
# 가장 많이 헷갈린 짝의 틀린 낙서 최대 8장
wrong_index = []
for i in range(len(y_test)):
    if y_test[i].item() == top_true and test_predictions[i].item() == top_pred:
        wrong_index.append(i)

fig, axes = plt.subplots(1, 8, figsize=(16, 2.8))
for k in range(8):
    axes[k].axis("off")
for k, i in enumerate(wrong_index[:8]):
    axes[k].imshow(X_test[i, 0], cmap="gray_r")
fig.suptitle(f"정답 {class_names[top_true]}인데 {class_names[top_pred]}(으)로 예측한 낙서", fontsize=14)
plt.tight_layout()
plt.show()

**결과 읽기**
- **별을 비행기로 예측한 17장:** 가장 많이 헷갈린 짝입니다(별 테스트 200장의 8.5%).
- **비행기를 별로 예측한 14장:** 반대 방향으로도 헷갈립니다. 두 종류는 서로를 착각하는 짝입니다.
- **고양이 169장, 집 194장:** 종류별로 맞힌 수가 가장 적은 종류는 고양이이고 가장 많은 종류는 집입니다(각 200장 중).

## 8. 모델을 저장하고 다시 불러오기

**배경**: 학습한 모델을 다른 노트북이나 서비스에서 쓰려면, 교안 01처럼 학습된 파라미터(`state_dict`)를 파일로 저장해 두고 같은 구조의 새 모델에 다시 채웁니다. 불러온 모델이 저장 전 모델과 같은 예측을 하는지 확인합니다.

**요구사항**
- (1) `final_model`의 `state_dict`를 `output_dir / "quickdraw_cnn.pt"`(**`model_path`**)에 저장합니다.
- (2) 저장한 파일을 읽은 **`loaded_state`**(`map_location`은 `device`)를 만듭니다.
- (3) `make_filter_cnn(final_filters)`로 만든 새 모델에 `loaded_state`를 채운 **`loaded_model`** 을 만듭니다.
- (4) `loaded_model`로 구한 `X_test`의 예측 번호(CPU 텐서) **`loaded_predictions`** 를 만듭니다.
- (5) `test_predictions`와 같은 예측의 개수를 파이썬 정수 **`same_count`** 로 만듭니다.

**확인 기준**: `model_path` 파일이 있고, `loaded_state`가 `final_model`의 `state_dict`를 그대로 담은 것이 아니라 파일에서 읽은 값이며 그 값이 `final_model`의 파라미터와 같습니다. `loaded_model`은 `final_model`과 다른 객체이면서 파라미터가 같고, `same_count`가 2,000입니다.

<details><summary>힌트</summary>

```text
접근방법:
- 저장은 torch.save에 저장할 값과 파일 경로를 넘깁니다. 불러오기는 torch.load로 읽은 값을 같은 구조의 새 모델에 채웁니다.

세부구현:
1. 모델 객체가 아니라 state_dict를 저장합니다.
2. 새 모델은 load_state_dict를 부르기 전에는 학습 전 파라미터 그대로입니다.
3. 두 예측 텐서를 == 로 비교하면 칸마다 True 또는 False가 나옵니다. sum으로 True의 개수를 세고 item으로 파이썬 정수를 꺼냅니다.
```

</details>

In [ ]:
# [작성] 최종 모델을 저장하고, 같은 구조의 새 모델로 불러와 예측을 비교하세요.

# 학습된 파라미터를 파일로 저장
model_path = output_dir / "quickdraw_cnn.pt"
...

# 저장한 파일을 읽습니다
loaded_state = ...

# 같은 구조의 새 모델에 읽은 값을 채웁니다
loaded_model = ...
...

# 불러온 모델의 예측과 저장 전 예측 비교
with torch.no_grad():
    loaded_predictions = ...
same_count = ...
print("저장 파일:", model_path.name, "| 같은 예측:", same_count, "/", len(test_predictions))

In [ ]:
# [자가채점] 저장 파일, 불러온 파라미터, 예측을 확인합니다.
assert model_path.exists() and model_path.stat().st_size > 0, f"{model_path} 파일이 없습니다. final_model의 state_dict를 model_path에 저장했는지 확인하세요."

# 읽은 값: 파라미터 이름과 값이 final_model과 같아야 합니다
assert isinstance(loaded_state, dict) and "0.weight" in loaded_state, "loaded_state가 state_dict가 아닙니다. 모델 객체 자체가 아니라 final_model의 state_dict를 저장하고, 그 파일을 읽어 담으세요."
final_state = final_model.state_dict()
assert list(loaded_state) == list(final_state), "loaded_state의 파라미터 이름이 final_model과 다릅니다. final_model의 state_dict를 저장했는지 확인하세요."
for key in final_state:
    assert torch.equal(loaded_state[key], final_state[key]), f"loaded_state의 {key} 값이 final_model과 다릅니다. 학습을 마친 final_model의 state_dict를 저장하세요."

# 파일에서 읽은 값인지: 파일에서 읽은 텐서는 final_model의 파라미터와 다른 메모리에 있습니다
assert loaded_state["0.weight"].data_ptr() != final_model[0].weight.data_ptr(), "loaded_state가 final_model의 state_dict를 그대로 담은 것입니다. 저장한 파일을 읽어 담으세요."

assert isinstance(loaded_model, nn.Sequential) and loaded_model is not final_model, "loaded_model은 make_filter_cnn(final_filters)로 새로 만든 모델입니다. final_model을 그대로 담지 마세요."
assert torch.equal(loaded_model[0].weight, final_model[0].weight) and torch.equal(loaded_model[7].weight, final_model[7].weight), "loaded_model의 파라미터가 학습 전 값입니다. loaded_state를 loaded_model에 채웠는지 확인하세요."

assert isinstance(same_count, int), "same_count는 파이썬 정수입니다. 개수를 센 결과에서 숫자를 꺼냈는지 확인하세요."
assert same_count == 2000, f"같은 예측이 {same_count}개입니다. 불러온 모델로 X_test 전체를 예측해 test_predictions와 비교하세요."
print("✅ 8번 통과")

**결과 읽기**
- **2000 / 2000:** 불러온 모델의 예측이 저장 전 모델의 예측과 모두 같습니다.
- **파라미터만 저장:** 불러올 때는 같은 구조의 모델(`make_filter_cnn(final_filters)`)을 먼저 만들어야 합니다.

## 9. 합성곱 층과 Linear 층의 파라미터 수 비교하기

**배경**: 교안 01 2-6의 지역 연결(출력 한 칸이 입력의 작은 구역만 봄)과 파라미터 공유(같은 필터를 모든 자리에서 씀) 때문에 합성곱 층은 파라미터가 적습니다. 4번에서 학습한 `model`의 둘째 합성곱 층과 입출력 크기가 같은 `nn.Linear`를 만들어 두 층의 파라미터 수를 세어 비교합니다.

**요구사항**
- (1) `model`의 둘째 합성곱 층(번호 3)을 **`conv_layer`** 에 담습니다.
- (2) 3번의 학습 미니배치 `X_batch`를 계산 장치로 옮겨(3번처럼 `.to(device)`) `model`의 앞 세 층(합성곱, ReLU, 풀링)에 통과시킨 둘째 합성곱 층의 입력 **`conv_input`**(64, 16, 14, 14)을 만듭니다.
- (3) `conv_input`을 `conv_layer`에 넣은 출력 **`conv_output`**(64, 32, 14, 14)을 만듭니다.
- (4) 그림 한 장의 입력 값 개수(`conv_input[0]`의 값 개수)를 받아 출력 값 개수(`conv_output[0]`의 값 개수)를 내는 `nn.Linear`를 **`linear_layer`** 에 담습니다. 두 개수는 텐서의 `numel()`로 구합니다.
- (5) `conv_layer`의 파라미터(가중치와 절편) 값 개수를 모두 더한 정수 **`conv_params`** 를 만듭니다.
- (6) 같은 방법으로 `linear_layer`의 파라미터 수 **`linear_params`** 를 만듭니다.
- (7) `linear_params`를 `conv_params`로 나눈 **`param_ratio`** 를 만듭니다.

**예시**: 첫 합성곱 층 `Conv2d(1, 16, kernel_size=3)`의 파라미터는 가중치 16 x 1 x 3 x 3 = 144개와 절편 16개, 모두 160개입니다.

**확인 기준**: `conv_layer`가 `model`의 둘째 합성곱 층이고 `conv_input`, `conv_output`의 모양이 위와 같습니다. `linear_layer`의 입력 크기와 출력 크기가 두 텐서의 그림 한 장 값 개수와 같고, `conv_params`, `linear_params`가 가중치와 절편을 모두 센 정수이며, `param_ratio`가 두 값의 비율입니다.

<details><summary>힌트</summary>

```text
접근방법:
- 층을 번호로 꺼내 입력을 통과시키고, 텐서의 numel로 크기를, 층의 parameters로 파라미터 수를 셉니다.

세부구현:
1. nn.Sequential은 리스트처럼 번호로 층을 꺼내고, 잘라서 앞 몇 층만 쓸 수 있습니다(교안 01 5-3).
2. 학습이 아니므로 층을 통과시킬 때 기울기 기록을 끕니다.
3. numel은 텐서에 든 값의 개수를 돌려줍니다. 그림 한 장은 첫 축의 0번입니다.
4. 층의 parameters는 가중치와 절편 텐서를 차례로 돌려줍니다. 교안 01 5-3에서 잔차 블록의 파라미터 수를 센 반복문과 같은 방법입니다.
```

</details>

In [ ]:
# [작성] 둘째 합성곱 층과, 입출력 크기가 같은 Linear 층의 파라미터 수를 비교하세요.

# 1. 둘째 합성곱 층과 그 층의 입력, 출력
conv_layer = ...
with torch.no_grad():
    conv_input = ...
    conv_output = ...
print("합성곱 층 입력:", tuple(conv_input.shape), "| 출력:", tuple(conv_output.shape))

# 2. 그림 한 장의 입력 값 개수를 받아 출력 값 개수를 내는 Linear 층
linear_layer = ...
print(linear_layer)

# 3. 두 층의 파라미터 수(가중치와 절편)
conv_params = 0
...
linear_params = 0
...
param_ratio = ...
print(f"합성곱 층: {conv_params:,}개 | Linear 층: {linear_params:,}개 | 배율: {param_ratio:,.1f}")

In [ ]:
# [자가채점] 고른 층, 입력과 출력의 모양, Linear 층의 크기, 파라미터 수를 확인합니다.
assert conv_layer is model[3], "conv_layer는 model의 둘째 합성곱 층(번호 3)입니다. 첫 합성곱 층(번호 0)이 아닌지 확인하세요."
assert isinstance(conv_input, torch.Tensor) and tuple(conv_input.shape) == (64, 16, 14, 14), f"conv_input의 모양이 {tuple(conv_input.shape)}입니다. (64, 16, 14, 14)여야 합니다. X_batch를 model의 앞 세 층(합성곱, ReLU, 풀링)에만 통과시켰는지 확인하세요."
assert isinstance(conv_output, torch.Tensor) and tuple(conv_output.shape) == (64, 32, 14, 14), f"conv_output의 모양이 {tuple(conv_output.shape)}입니다. conv_input을 conv_layer 하나에만 넣으세요."

# Linear 층: 그림 한 장의 입력 값 개수 -> 출력 값 개수
assert isinstance(linear_layer, nn.Linear), "linear_layer는 nn.Linear입니다. `...`를 채웠는지 확인하세요."
assert linear_layer.in_features == conv_input[0].numel(), f"linear_layer의 입력 크기({linear_layer.in_features})가 그림 한 장의 입력 값 개수({conv_input[0].numel()})와 다릅니다. 입력 크기 자리에 conv_input[0]의 값 개수를 넣으세요."
assert linear_layer.out_features == conv_output[0].numel(), f"linear_layer의 출력 크기({linear_layer.out_features})가 그림 한 장의 출력 값 개수({conv_output[0].numel()})와 다릅니다."
assert linear_layer.bias is not None, "linear_layer는 절편이 있는 기본 nn.Linear입니다."

# 파라미터 수: 가중치와 절편을 모두 센 정수
assert isinstance(conv_params, int) and isinstance(linear_params, int), "conv_params, linear_params는 파이썬 정수입니다. 0에서 시작해 numel()을 더했는지 확인하세요."
assert conv_params == 4640, f"conv_params가 {conv_params:,}개입니다. conv_layer의 parameters()가 돌려주는 텐서(가중치, 절편)의 값 개수를 모두 더했는지 확인하세요."
assert linear_params == 19675264, f"linear_params가 {linear_params:,}개입니다. linear_layer의 parameters()가 돌려주는 텐서(가중치, 절편)의 값 개수를 모두 더했는지 확인하세요."
assert 4200 < param_ratio < 4300, f"param_ratio가 {param_ratio:.1f}입니다. linear_params를 conv_params로 나눈 값입니다."
print("✅ 9번 통과")

**결과 읽기**
- **합성곱 층 4,640개, Linear 층 19,675,264개:** 입출력 크기가 같아도 `nn.Linear`의 파라미터가 약 4,240배 많습니다.
- **(16, 14, 14)에서 (32, 14, 14):** `nn.Linear`는 출력 6,272칸이 저마다 입력 3,136개 모두와 이어지고, 합성곱 층은 출력 한 칸이 3 x 3 x 16 = 144개만 보며(지역 연결) 같은 필터 32개를 14 x 14 = 196자리에서 함께 씁니다(파라미터 공유).

## 10. 배치 정규화를 넣은 CNN과 학습 손실 비교하기

**배경**: 교안 01 5-2의 배치 정규화는 층 출력의 크기를 미니배치마다 맞춰 학습을 안정시킵니다. 5번과 같은 앞 3,000장, 같은 5에폭으로 합성곱마다 배치 정규화를 넣은 CNN을 학습해 필터 16개 후보(`filter_curves[16]`)와 학습 손실을 비교합니다. 제공 함수 `train_one_epoch`, `evaluate`는 모델의 모드를 바꾸지 않으므로, 배치 정규화가 든 모델은 학습 앞과 평가 앞에 모드를 직접 바꿉니다.

**요구사항**
- (1) 필터 수를 받아, 3번 표의 CNN에서 두 합성곱 층 바로 뒤에 `nn.BatchNorm2d(그 합성곱의 출력 채널 수)`를 하나씩 넣은 모듈 10개짜리 CNN을 돌려주는 함수 **`make_bn_cnn(n_filters)`** 를 만듭니다. 층 순서는 `Conv2d`, `BatchNorm2d`, `ReLU`, `MaxPool2d`를 두 번, 이어서 `Flatten`, `Linear`이고, 맨 앞에서 시드 0을 고정하고 계산 장치에 올립니다.
- (2) `make_bn_cnn(16)`을 담은 **`bn_model`** 을 만듭니다.
- (3) `bn_model`의 파라미터를 학습률 0.001로 갱신하는 Adam 옵티마이저 **`bn_optimizer`** 를 만듭니다.
- (4) `num_epochs`번 반복하며, 에폭마다 `bn_model`을 학습 모드로 바꾼 뒤 `train_one_epoch`로 `subset_loader`를 한 번 학습합니다.
- (5) 이어서 평가 모드로 바꾼 뒤 `evaluate`로 `subset_eval_loader`(제공 줄, 섞지 않음)의 정확도를 구합니다.
- (6) 에폭마다 `에폭`(1부터), `학습 손실`, `학습 데이터 정확도` 세 키의 dict를 `bn_rows`에 더합니다. 표 **`bn_history`** 는 골격이 만듭니다.
- (7) `모델`, `첫 에폭 학습 손실`, `마지막 학습 손실`, `마지막 학습 데이터 정확도` 네 키의 dict 두 개로 표 **`bn_compare_table`** 을 만듭니다. 첫 행은 `"원래 CNN"`(`filter_curves[16]`의 값), 둘째 행은 `"배치 정규화 CNN"`(`bn_history`의 값)입니다.

**예시**: `make_bn_cnn(8)`의 둘째 층은 `BatchNorm2d(8)`, 여섯째 층은 `BatchNorm2d(16)`입니다.

**확인 기준**: `make_bn_cnn(8)`이 위 순서의 모듈 10개이고 배치 정규화의 채널 수가 바로 앞 합성곱의 출력 채널 수와 같으며, 두 번 불러도 첫 층 가중치가 같습니다. `bn_model`의 배치 정규화 층이 학습 모드에서 지난 미니배치 수(`num_batches_tracked`)가 5에폭의 학습 스텝 수(235)와 같고(평가할 때 학습 모드이면 더 많아지고, 다음 에폭을 평가 모드로 학습하면 더 적어집니다), 마지막 학습 데이터 정확도가 평가 모드의 `bn_model`로 앞 3,000장을 다시 평가한 값과 같습니다. `bn_compare_table`의 두 행이 `filter_curves[16]`, `bn_history`의 값과 같고, 배치 정규화 CNN의 마지막 학습 손실이 원래 CNN보다 작습니다.

<details><summary>힌트</summary>

```text
접근방법:
- 4번 train_cnn 안의 에폭 반복문과 같은 모양에서, 학습 앞과 평가 앞에 모드를 바꾸는 줄을 하나씩 더합니다.

세부구현:
1. make_bn_cnn은 3번 make_filter_cnn에서 두 Conv2d 바로 뒤에 BatchNorm2d를 한 줄씩 더한 모양입니다. 배치 정규화는 ReLU 앞에 둡니다.
2. 모드는 교안 01 5-2의 두 메서드로 바꿉니다. evaluate 안의 torch.no_grad()는 기울기 기록만 끄고 층의 동작은 바꾸지 않습니다.
3. 평가 모드로 바꾼 채 다음 에폭을 학습하면, 배치 정규화가 이번 미니배치 대신 이동 평균으로 표준화하고 이동 평균도 갱신하지 않습니다.
4. 원래 CNN 행의 값은 filter_curves[16] 표에서 학습 손실 열의 첫 줄과 마지막 줄, 학습 데이터 정확도 열의 마지막 줄입니다.
```

</details>

In [ ]:
# [작성] (1) 합성곱마다 배치 정규화를 넣은 CNN 함수를 완성하세요.
def make_bn_cnn(n_filters):
    """make_filter_cnn(n_filters)의 두 합성곱 층 바로 뒤에 배치 정규화를 넣은 CNN을 새로 만들어 돌려줍니다."""
    ...


print(make_bn_cnn(16))

In [ ]:
# [작성] (2)~(6) 배치 정규화 CNN을 앞 3,000장으로 학습하며 학습 손실과 학습 데이터 정확도를 기록하세요.

# 학습 데이터 정확도를 계산할 DataLoader(제공): 앞 3,000장을 섞지 않고 읽습니다
subset_eval_loader = DataLoader(subset_loader.dataset, batch_size=64, shuffle=False)

# 새 모델과 옵티마이저
bn_model = ...
bn_optimizer = ...

# 에폭마다 학습 모드로 학습하고, 평가 모드로 학습 데이터 정확도를 계산합니다
bn_rows = []
for epoch in range(num_epochs):
    ...

bn_history = pd.DataFrame(bn_rows)
display(bn_history.set_index("에폭").round(3))

In [ ]:
# [작성] (7) 원래 CNN(필터 16개)과 배치 정규화 CNN의 기록을 한 표로 모으세요.
bn_compare_table = pd.DataFrame([
    ...,  # 원래 CNN의 행(dict)
    ...,  # 배치 정규화 CNN의 행(dict)
])
display(bn_compare_table.round(3))

In [ ]:
# [자가채점] 층 순서, 모드 전환, 기록, 비교 표를 확인합니다.
bn_a = make_bn_cnn(8)
assert isinstance(bn_a, nn.Sequential) and len(bn_a) == 10, "make_bn_cnn()은 모듈 10개짜리 nn.Sequential을 돌려줍니다. `...`를 채웠는지, 함수 끝에서 모델을 돌려주는지 확인하세요."
layer_names = [type(layer).__name__ for layer in bn_a]
expected_names = ["Conv2d", "BatchNorm2d", "ReLU", "MaxPool2d", "Conv2d", "BatchNorm2d", "ReLU", "MaxPool2d", "Flatten", "Linear"]
assert layer_names == expected_names, f"층 순서가 {layer_names}입니다. 배치 정규화는 합성곱 바로 뒤, ReLU 앞에 둡니다(교안 01 5-2)."
assert bn_a[1].num_features == 8 and bn_a[5].num_features == 16, "BatchNorm2d의 채널 수는 바로 앞 합성곱의 출력 채널 수(n_filters, n_filters의 2배)입니다."
assert bn_a[0].out_channels == 8 and bn_a[4].out_channels == 16 and bn_a[9].in_features == 16 * 7 * 7, "합성곱과 출력층의 크기는 3번 make_filter_cnn과 같습니다."
assert torch.equal(bn_a[0].weight, make_bn_cnn(8)[0].weight), "make_bn_cnn()을 두 번 부르면 첫 층 가중치가 같아야 합니다. 모델을 만들기 직전에 시드 0을 고정했는지 확인하세요."
assert next(bn_a.parameters()).device.type == device.type, "모델을 계산 장치에 올렸는지 확인하세요."

# 모델과 옵티마이저
assert isinstance(bn_model, nn.Sequential) and len(bn_model) == 10 and bn_model[0].out_channels == 16, "bn_model은 make_bn_cnn(16)으로 만든 모델입니다."
assert isinstance(bn_optimizer, torch.optim.Adam) and bn_optimizer.param_groups[0]["lr"] == 0.001, "bn_optimizer는 학습률 0.001인 torch.optim.Adam입니다."
assert bn_optimizer.param_groups[0]["params"][0] is bn_model[0].weight, "bn_optimizer는 bn_model의 파라미터로 만드세요."

# 기록 표
assert isinstance(bn_history, pd.DataFrame) and list(bn_history.columns) == ["에폭", "학습 손실", "학습 데이터 정확도"], "bn_rows의 dict는 에폭, 학습 손실, 학습 데이터 정확도 세 키를 이 순서로 가집니다."
assert bn_history["에폭"].tolist() == list(range(1, num_epochs + 1)), "bn_history는 에폭마다 한 줄, 에폭 번호는 1부터 5까지입니다."

# 모드 전환: 배치 정규화 층은 학습 모드에서 미니배치를 하나 지날 때마다 num_batches_tracked를 1씩 늘립니다
train_steps = num_epochs * len(subset_loader)
tracked = bn_model[1].num_batches_tracked.item()
assert tracked <= train_steps, f"배치 정규화 층이 학습 모드에서 지난 미니배치가 {tracked}개로, 학습 스텝 수 {train_steps}보다 많습니다. 정확도를 계산할 때도 학습 모드여서 이동 평균이 평가 중에도 바뀌었습니다. evaluate를 부르기 전에 평가 모드로 바꾸세요."
assert tracked >= train_steps, f"배치 정규화 층이 학습 모드에서 지난 미니배치가 {tracked}개로, 학습 스텝 수 {train_steps}보다 적습니다. 평가 모드로 바꾼 뒤 다음 에폭을 학습하기 전에 학습 모드로 되돌렸는지 확인하세요."

# 기록한 정확도: 평가 모드의 bn_model로 앞 3,000장을 다시 평가한 값과 같아야 합니다
bn_model.eval()
_, recheck_acc = evaluate(bn_model, DataLoader(subset_loader.dataset, batch_size=64), loss_fn)
recorded_acc = bn_history["학습 데이터 정확도"].iloc[-1]
assert abs(recorded_acc - recheck_acc) < 0.005, f"마지막 학습 데이터 정확도({recorded_acc:.4f})가 평가 모드의 bn_model로 앞 3,000장을 다시 평가한 값({recheck_acc:.4f})과 다릅니다. 평가 모드로 바꾼 뒤 subset_eval_loader로 평가했는지 확인하세요."

# 비교 표: 원래 CNN은 5번의 필터 16개 후보입니다
assert isinstance(bn_compare_table, pd.DataFrame) and list(bn_compare_table.columns) == ["모델", "첫 에폭 학습 손실", "마지막 학습 손실", "마지막 학습 데이터 정확도"], "bn_compare_table의 열은 모델, 첫 에폭 학습 손실, 마지막 학습 손실, 마지막 학습 데이터 정확도입니다."
assert bn_compare_table["모델"].tolist() == ["원래 CNN", "배치 정규화 CNN"], "행은 원래 CNN, 배치 정규화 CNN 순서입니다."
for k, curve in enumerate([filter_curves[16], bn_history]):
    row = bn_compare_table.iloc[k]
    assert abs(row["첫 에폭 학습 손실"] - curve["학습 손실"].iloc[0]) < 1e-9 and abs(row["마지막 학습 손실"] - curve["학습 손실"].iloc[-1]) < 1e-9, f"{row['모델']} 행의 학습 손실이 기록 표와 다릅니다. 원래 CNN은 filter_curves[16], 배치 정규화 CNN은 bn_history의 첫 줄과 마지막 줄입니다."
    assert abs(row["마지막 학습 데이터 정확도"] - curve["학습 데이터 정확도"].iloc[-1]) < 1e-9, f"{row['모델']} 행의 마지막 학습 데이터 정확도가 기록 표의 마지막 줄 값과 다릅니다."
plain_last = bn_compare_table["마지막 학습 손실"].iloc[0]
bn_last = bn_compare_table["마지막 학습 손실"].iloc[1]
assert bn_last < plain_last, f"배치 정규화 CNN의 마지막 학습 손실({bn_last:.3f})이 원래 CNN({plain_last:.3f})보다 작지 않습니다. 배치 정규화 층의 위치와 모드 전환을 확인하세요."
print("✅ 10번 통과")

In [ ]:
# 원래 CNN과 배치 정규화 CNN의 에폭별 학습 손실
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(filter_curves[16]["에폭"], filter_curves[16]["학습 손실"], marker="o", label="원래 CNN (필터 16개)")
ax.plot(bn_history["에폭"], bn_history["학습 손실"], marker="o", label="배치 정규화 CNN (필터 16개)")
ax.set_xticks(range(1, num_epochs + 1))
ax.set_xlabel("에폭", fontsize=13)
ax.set_ylabel("학습 손실", fontsize=13)
ax.set_title("학습 손실 (앞 3,000장)", fontsize=14)
ax.tick_params(labelsize=11)
ax.legend(loc="upper right", fontsize=11)
plt.tight_layout()
plt.show()

**결과 읽기**
- **첫 에폭 학습 손실 1.662와 1.090:** 같은 데이터, 같은 에폭인데 배치 정규화 CNN이 첫 에폭부터 학습 데이터에 더 빨리 맞춥니다.
- **마지막 학습 손실 0.442와 0.219, 학습 데이터 정확도 0.871과 0.950:** 5에폭 뒤 학습 손실이 절반 아래입니다. 층 출력의 크기를 맞춰 학습을 안정시키는 배치 정규화의 목적과 같은 결과입니다.
- **학습 데이터로 계산한 값:** 처음 보는 낙서의 정확도까지 높은지는 이 표로 알 수 없습니다. 학습에 쓰지 않은 데이터로 두 모델을 비교하는 검증 데이터는 교안 02에 있습니다.